# Lähde-ekstraktio: {Author Year}

DOI: `{doi}`  
PDF: `data/raw/{path/to/pdf}`

Tämä notebook tekee viisi vaihetta:

1. **Lukee PDF:n** (avaa Claude Code -sivupaneelista)
2. **LLM-raakaekstraktio** Pydantic-skeemalla `RawPair`
3. **Rikastaa** SMILES:t PubChem:istä (RDKit kanonisoi)
4. **Soveltaa** `classify_baird_taylor`-luokitusta
5. **Validoi** mole/weight-summat, SMILES, säilytysolot

## Periaate: LLM ei päätä luokituksia

Älä **koskaan** editoi `gfa_class`- tai `label_confidence`-kenttiä käsin. Jos rivi näyttää väärin luokitellulta, korjaa sen syöttötiedot (esim. `induction_time_censored`, `storage_T_C`, `experimental_protocol`) ja anna luokituksen päivittyä automaattisesti, kun ajat notebookin uudelleen.

Sama koskee SMILES-merkkijonoja: ne tulevat PubChem:istä RDKit-kanonisoinnin kautta, ei LLM:n muistista. Jos PubChem ei tunnista nimeä, rivi merkitään `needs_review=True` ja korjataan käsin `*_raw.json`-tiedostoon.

## Solu 2 — Imports ja konfiguraatio

In [6]:
import json
from pathlib import Path

import pandas as pd

from coamorphous.corpus.schema import build_schema, load_schema_yaml
from coamorphous.extraction.enrich import (
    assign_pair_id,
    raw_pair_to_master_row,
)
from coamorphous.extraction.extraction_schema import RawPair
from coamorphous.extraction.validate import run_all_validations

# --- Säädettävät parametrit per paperi ---------------------------------------
# Korvaa nämä jokaisen lähteen kohdalla.
PDF_PATH = Path("data/raw/lobmann_2012.pdf")
SOURCE_DOI = "10.1016/j.ijpharm.2012.05.016"
SOURCE_FIRST_AUTHOR = "lobmann"   # esim. "fink"
SOURCE_YEAR = 2012
EXTRACTED_BY = "claude_code"

# Polut, jotka johdetaan automaattisesti yllä olevista parametreista.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_JSON_PATH = PROJECT_ROOT / "data" / "interim" / f"{SOURCE_FIRST_AUTHOR}_{SOURCE_YEAR}_raw.json"
INTERIM_CSV_PATH = PROJECT_ROOT / "data" / "interim" / f"{SOURCE_FIRST_AUTHOR}_{SOURCE_YEAR}.csv"
SCHEMA_YAML = PROJECT_ROOT / "configs" / "corpus_schema.yaml"

print(f"PDF: {PDF_PATH}")
print(f"Raw JSON output: {RAW_JSON_PATH}")
print(f"Interim CSV output: {INTERIM_CSV_PATH}")

PDF: data\raw\lobmann_2012.pdf
Raw JSON output: c:\Users\perhe\vscode\coamorphous\data\interim\lobmann_2012_raw.json
Interim CSV output: c:\Users\perhe\vscode\coamorphous\data\interim\lobmann_2012.csv


## Vaihe 1 — LLM-raakaekstraktio (automatisoitu)

Tämä solu kutsuu **Claude Code -CLI:tä** subprocessina, joka:

1. Lukee PDF:n (`PDF_PATH`)
2. Ekstraktoi co-amorfiset parit `RawPair`-skeeman mukaisesti
3. Tallentaa raaka-JSONin polkuun `RAW_JSON_PATH`

Aikaisemmin tämä vaihe vaati promptin manuaalisen kopioinnin Claude Coden sivupaneeliin. Nyt ekstraktio ajetaan kerralla notebookin sisältä — sama LLM, sama prompt, mutta ilman copy-paste-vaihetta.

### Vaatimukset

- **`claude` CLI** asennettu globaalisti:
  ```bash
  npm install -g @anthropic-ai/claude-code
  ```
- **Sisäänkirjautuminen Max-tilauksellasi**. Jos et ole vielä kirjautunut CLI:llä, aja terminaalissa kerran `claude` ja seuraa OAuth-prosessia. CLI ja VS Code -laajennus voivat käyttää eri credentialeja, joten myös CLI vaatii oman kirjautumisen.
- **PDF saatavilla** polussa `PDF_PATH`.

### Käyttö

Aja alla oleva solu kun olet päivittänyt `PDF_PATH` ja muut parametrit Solu 2:ssa. Ekstraktio kestää tyypillisesti 30–90 sekuntia per PDF; pidemmissä artikkeleissa pidempään.

### Käytetty prompt (auditointia varten)

Sama promptirunko ajetaan jokaiselle PDF:lle, vain `PDF_PATH` ja `RAW_JSON_PATH` substituoidaan. Promptin sisältö on koodissa alla; tässä se näytetään vertailukohdaksi:

```
Lue PDF: {PDF_PATH}

Ekstraktoi KAIKKI ko-amorfiset parit RawPair-skeeman mukaisesti
(src/coamorphous/extraction/extraction_schema.py).

KRIITTISET SÄÄNNÖT:
1. Älä päättele tai laske kanonisia SMILES:eja, InChIKeyjä, gfa_class:ia,
   label_confidence:ia tai mitään pari-tason deskriptoreita - ne lasketaan
   automaattisesti.
2. Anna ratio_source_quote eksakti lainaus lähteestä (esim. "1:1 mol"
   tai "70:30 w/w").
3. Jos suhteita on annettu massoina, anna molemmat (weight_fraction ja
   mole_fraction laskettuna massoista, KUNNES Tm:t ovat tiedossa).
4. Aseta induction_time_censored TARKASTI:
   - True = kokeilu loppui ilman havaittua kiteytymistä
   - False = kiteytyminen havaittiin annetulla ajanhetkellä
5. Aseta experimental_protocol parhaalla mahdollisella tarkkuudella:
   - 40 °C / 75 % RH -> ich_q1a_accelerated
   - 25 °C / 60 % RH -> ich_q1a_long_term
   - Kuiva (P2O5/silica), <30 vrk -> dry_short_term
   - Tg+15 K -mittaus -> tg_plus_15K
   - Pelkkä DSC-uusinta -> dsc_in_situ
   - Muu / epäselvä -> non_standard
6. Aseta needs_review=True ja review_reasons-listalle, jos:
   - Lähde on epäjohdonmukainen
   - Sensurointi on epäselvä
   - Protokolla ei sovi mihinkään selvästi
   - Stabiiliuskoetta ei raportoitu lainkaan (vain DSC tai PXRD)
7. source_quote on 1-2 lauseen lainaus lähteestä, joka tukee tämän parin tietoja.

Tallenna lopputulos JSON-listana tiedostoon: {RAW_JSON_PATH}
```

### Manuaalinen vararatkaisu

Jos automaattinen ajo epäonnistuu (esim. CLI ei ole kirjautunut, PDF on liian iso), voit edelleen avata PDF:n VS Codessa rinnakkain ja kopioida promptin Claude Code -sivupaneeliin manuaalisesti.

In [7]:
import shutil
import subprocess

# --- 1) Tarkista että claude CLI on PATH:issa --------------------------------
# Jos tämä epäonnistuu, asenna globaalisti: npm install -g @anthropic-ai/claude-code
claude_bin = shutil.which("claude")
if claude_bin is None:
    raise RuntimeError(
        "claude CLI:tä ei löydy PATH:ista. Asenna: "
        "npm install -g @anthropic-ai/claude-code"
    )
print(f"Käytetään claude CLI: {claude_bin}")

# --- 2) Muodosta promptirunko substituoiduilla poluilla ----------------------
# Käytetään suhteellisia polkuja projektin juuresta (Claude operoi cwd=PROJECT_ROOT),
# jotta se voi lukea PDF:n ja kirjoittaa JSONin oman --add-dir-laajuutensa sisällä.
pdf_relative = (
    PDF_PATH.relative_to(PROJECT_ROOT) if PDF_PATH.is_absolute() else PDF_PATH
)
raw_json_relative = RAW_JSON_PATH.relative_to(PROJECT_ROOT)

# System prompt: kerrotaan agentin rooli kerralla, jotta se ei tulkitse promptia
# yhteenvetokutsuksi tai kysymykseksi vaan tehtäväksi joka päättyy Write-toolin
# kutsuun ja lyhyeen vahvistukseen.
system_prompt = (
    "Olet data-ekstraktioagentti coamorphous-tutkimusprojektissa. "
    "Tehtäväsi on YKSI: lue annettu PDF, muodosta RawPair-skeeman mukainen "
    "JSON-lista co-amorfisista pareista, ja TALLENNA se Write-toolilla "
    "annettuun polkuun. ÄLÄ anna PDF:n yhteenvetoja, ÄLÄ kysy lisätietoja, "
    "ÄLÄ ehdota vaihtoehtoja. Käytä Read PDF:n lukemiseen ja Write tallennukseen. "
    "Lopuksi vastaa pelkästään yhdellä rivillä: '[OK] Tallennettu N paria tiedostoon X.'"
)

# Imperatiivinen prompti: TEHTÄVÄ-otsikko ja numeroidut vaiheet, jotta Claude
# ei tulkitse tätä keskusteluksi vaan suoritettavaksi työnkuluksi.
prompt = f"""TEHTÄVÄ: Ekstraktoi co-amorfiset parit PDF:stä JSON-tiedostoon.

INPUT PDF:           {pdf_relative}
OUTPUT JSON:         {raw_json_relative}
SKEEMA:              src/coamorphous/extraction/extraction_schema.py (RawPair)

VAIHEET (suorita JÄRJESTYKSESSÄ, älä kysy lupaa):
1. Lue PDF Read-toolilla (polku yllä).
2. Tunnista taulukoista ja tekstistä KAIKKI co-amorfiset parit (eri suhteet =
   eri rivit, eri säilytysolot = eri rivit).
3. Muodosta jokaisesta parista RawPair-skeeman mukainen dict KRIITTISTEN
   SÄÄNTÖJEN mukaan (alla).
4. Kirjoita Write-toolilla kaikki parit JSON-listana yllä mainittuun OUTPUT-polkuun.
5. Vahvistus: vastaa pelkkä rivi '[OK] Tallennettu N paria tiedostoon: <polku>.'

KRIITTISET SÄÄNNÖT (sovella vaiheessa 3):
1. Älä päättele tai laske kanonisia SMILES:eja, InChIKeyjä, gfa_class:ia,
   label_confidence:ia tai mitään pari-tason deskriptoreita - ne lasketaan
   automaattisesti myöhemmin.
2. Anna ratio_source_quote eksakti lainaus lähteestä (esim. "1:1 mol"
   tai "70:30 w/w").
3. Jos suhteita on annettu massoina, anna molemmat (weight_fraction ja
   mole_fraction laskettuna massoista, KUNNES Tm:t ovat tiedossa).
4. Aseta induction_time_censored TARKASTI:
   - True = kokeilu loppui ilman havaittua kiteytymistä
   - False = kiteytyminen havaittiin annetulla ajanhetkellä
5. Aseta experimental_protocol parhaalla mahdollisella tarkkuudella:
   - 40 °C / 75 % RH -> ich_q1a_accelerated
   - 25 °C / 60 % RH -> ich_q1a_long_term
   - Kuiva (P2O5/silica), <30 vrk -> dry_short_term
   - Tg+15 K -mittaus -> tg_plus_15K
   - Pelkkä DSC-uusinta -> dsc_in_situ
   - Muu / epäselvä -> non_standard
6. Aseta needs_review=True ja review_reasons-listalle, jos:
   - Lähde on epäjohdonmukainen (esim. taulukon otsikko vs. tekstin numerot)
   - Sensurointi on epäselvä
   - Protokolla ei sovi mihinkään selvästi
   - Stabiiliuskoetta ei raportoitu lainkaan (vain DSC tai PXRD)
7. source_quote on 1-2 lauseen lainaus lähteestä, joka tukee tämän parin tietoja.

ÄLÄ anna yhteenvetoja, älä kysy tarkennuksia, älä luettele vaihtoehtoja.
Suorita vaiheet 1-5 ja vastaa vahvistuksella."""

# --- 3) Tyhjennä vanha JSON ennen ajoa ---------------------------------------
# Näin näemme luotettavasti onko Claude tällä ajolla luonut tiedoston.
if RAW_JSON_PATH.exists():
    RAW_JSON_PATH.unlink()
    print(f"Poistettu vanha: {RAW_JSON_PATH.name}")

# --- 4) Aja claude subprocessina ---------------------------------------------
# -p / --print                  = non-interactive print mode (tulosta vastaus + poistu)
# --append-system-prompt        = lisää extraktioagentti-rooli oletus-system-promptiin
# --allowedTools                = sallitaan vain Read (PDF) + Write (JSON), ei muita.
# --permission-mode bypassPermissions
#                               = -p-tilassa permission-promptit eivät toimi; rajaus
#                                 tapahtuu --allowedTools:lla.
cmd = [
    claude_bin,
    "-p",
    "--append-system-prompt", system_prompt,
    "--allowedTools", "Read,Write",
    "--permission-mode", "bypassPermissions",
    prompt,
]

print(f"\nKutsutaan Claude CLI:tä PDF:lle: {pdf_relative}")
print(f"Tulos kirjoitetaan: {raw_json_relative}")
print("Ekstraktio kestää tyypillisesti 30-90 s per PDF...\n")

result = subprocess.run(
    cmd,
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    timeout=900,  # 15 min hard cap
)

# --- 5) Näytä Claude'n vastaus debuggausta varten ----------------------------
if result.stdout:
    print("--- Claude vastaus ---")
    print(result.stdout)

# --- 6) Tarkista onnistuminen ------------------------------------------------
if result.returncode != 0:
    print("\n--- Stderr ---")
    print(result.stderr)
    raise RuntimeError(
        f"claude epäonnistui (exit code {result.returncode}). "
        f"Tarkista yllä oleva tulos."
    )

if not RAW_JSON_PATH.exists():
    raise RuntimeError(
        f"Claude ei luonut tiedostoa: {RAW_JSON_PATH}\n"
        f"Mahdollisia syitä: PDF:ää ei löytynyt, CLI ei kirjautunut, "
        f"prompt epäonnistui, tai Claude antoi yhteenvedon eikä kutsunut "
        f"Write-toolia. Tarkista yllä oleva vastaus."
    )

print(f"\n[OK] Tallennettu: {raw_json_relative}")
print("Jatka seuraavaan soluun (Pydantic-validointi).")

Käytetään claude CLI: C:\Users\perhe\AppData\Roaming\npm\claude.CMD

Kutsutaan Claude CLI:tä PDF:lle: data\raw\lobmann_2012.pdf
Tulos kirjoitetaan: data\interim\lobmann_2012_raw.json
Ekstraktio kestää tyypillisesti 30-90 s per PDF...

--- Claude vastaus ---
[OK] Tallennettu 15 paria tiedostoon data/interim/lobmann_2012_raw.json.


[OK] Tallennettu: data\interim\lobmann_2012_raw.json
Jatka seuraavaan soluun (Pydantic-validointi).


In [8]:
with open(RAW_JSON_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)

# Pydantic-validointi: kentät, tyypit, enumit, mole_fraction-rajat.
raw_pairs = [RawPair(**d) for d in raw_data]
print(f"Validoitu {len(raw_pairs)} paria.")

# Listaa heti needs_review-merkityt rivit, jotta ne näkyvät ennen rikastusta.
needs_review_now = [(i, p) for i, p in enumerate(raw_pairs) if p.needs_review]
if needs_review_now:
    print(f"\n[!] {len(needs_review_now)} paria merkitty needs_review jo ekstraktiossa:")
    for i, p in needs_review_now:
        print(f"  [{i}] {p.drug_A_name_raw}-{p.drug_B_name_raw}: {p.review_reasons}")

Validoitu 15 paria.

[!] 10 paria merkitty needs_review jo ekstraktiossa:
  [0] Simvastatin-Glipizide: ['Protokolla 4 °C / 0% RH (P2O5-desikkaattori), 128 d — ei sovi ICH-luokituksiin (kuiva mutta seuranta-aika >>30 vrk, ei dry_short_term); merkitty non_standard', 'XRPD/FTIR poikkeavat hieman (128 vs 133 d); käytetty XRPD bulk-kiteytymisen indikaattorina']
  [1] Simvastatin-Glipizide: ['Protokolla 25 °C / 0% RH (P2O5), 44 d — ei sovi standardiin (RH=0%, ei dry_short_term koska >30 vrk); merkitty non_standard', 'FTIR ja XRPD erivat merkittavasti (14 vrk vs 44 vrk); kerätty XRPD-arvo voi olla aliarvio bulk-kiteytymisen alkamisesta']
  [3] Simvastatin-Glipizide: ['Protokolla 4 °C / 0% RH (P2O5), 128 d — ei sovi ICH-luokituksiin; merkitty non_standard']
  [4] Simvastatin-Glipizide: ['Protokolla 25 °C / 0% RH (P2O5) ei sovi standardiin (RH=0%, >30 vrk); FTIR ja XRPD erivat (35 vrk vs 52 vrk)']
  [6] Simvastatin-Glipizide: ['Protokolla 4 °C / 0% RH (P2O5), 74 d — ei sovi ICH-luokituksiin; me

## Vaihe 2 — Rikasta + luokittele

`raw_pair_to_master_row` tekee per pari:

1. Hakee PubChem:istä SMILES + InChIKey + CID molemmille lääkkeille
2. RDKit-kanonisoinnin
3. `classify_baird_taylor`-luokituksen `experimental_protocol`-tietoinen
4. Yhdistää lähdemetadatan ja palauttaa kaikki 54 master-CSV:n saraketta

PubChem-haku ei ole instant — varaa muutama sekunti per pari.

In [9]:
source_metadata = {
    "source_doi": SOURCE_DOI,
    "source_first_author": SOURCE_FIRST_AUTHOR,
    "source_year": SOURCE_YEAR,
}

rows = [
    raw_pair_to_master_row(p, source_metadata, EXTRACTED_BY)
    for p in raw_pairs
]
rows = assign_pair_id(rows, SOURCE_FIRST_AUTHOR, SOURCE_YEAR)

df = pd.DataFrame(rows)
print(f"Rikastettu {len(df)} riviä, {len(df.columns)} saraketta.")
print("\nLuokat (gfa_class x label_confidence):")
print(df.groupby(["gfa_class", "label_confidence"], dropna=False).size())

Rikastettu 15 riviä, 54 saraketta.

Luokat (gfa_class x label_confidence):
gfa_class  label_confidence
2          high                 5
           low                 10
dtype: int64


## Vaihe 3 — Validoi

Kahdenlaisia tarkistuksia:

* **Riviä-kohti** (`run_all_validations`): mole/weight-summat, SMILES-validius, säilytysolojen yhteensopivuus protokollan kanssa.
* **Skeematason** (Pandera, `build_schema`): tyypit, enumit, vaaditut sarakkeet, sarakejärjestys.

Jos jompikumpi epäonnistuu, korjaa lähdön JSON ja aja Vaihe 2 uudelleen.

In [10]:
validation_errors = []
for idx, row in df.iterrows():
    errors = run_all_validations(row.to_dict())
    if errors:
        validation_errors.append((idx, row["pair_id"], errors))

if validation_errors:
    print(f"[!] {len(validation_errors)} riviä ei läpäissyt riviä-kohti -validointia:")
    for idx, pid, errs in validation_errors:
        print(f"  [{idx}] {pid}:")
        for e in errs:
            print(f"    - {e}")
else:
    print("[OK] Kaikki rivit läpäisivät riviä-kohti -validoinnin.")

# Pandera-skeemavalidointi yhdistetylle DataFramelle.
schema = build_schema(SCHEMA_YAML)
try:
    schema.validate(df, lazy=True)
    print("[OK] Pandera-skeema validoitu.")
except Exception as e:
    print(f"[!] Pandera-virhe:\n{e}")

[OK] Kaikki rivit läpäisivät riviä-kohti -validoinnin.
[OK] Pandera-skeema validoitu.


## Vaihe 4 — Tarkista needs_review / low_confidence -rivit

Listataan rivit, jotka vaativat ihmissilmää: joko LLM merkitsi `needs_review=True` ekstraktiossa, tai luokitin antoi `label_confidence='low'/'borderline'`.

In [11]:
# Master-CSV ei sisällä needs_review-saraketta (se on RawPair-tason kenttä),
# joten käytämme label_confidence- ja notes-kenttiä proxynä.
review_mask = df["label_confidence"].isin(["low", "borderline"]) | df["notes"].fillna("").str.contains(r"\[review\]", regex=True)
review_df = df[review_mask]

if len(review_df) > 0:
    print(f"Tarkista {len(review_df)} riviä manuaalisesti:")
    display_cols = [
        "pair_id", "drug_A_name", "drug_B_name",
        "gfa_class", "label_confidence", "experimental_protocol", "notes",
    ]
    print(review_df[display_cols].to_string())
else:
    print("[OK] Ei review-merkittyjä eikä low/borderline-rivejä.")

Tarkista 10 riviä manuaalisesti:
           pair_id  drug_A_name drug_B_name  gfa_class label_confidence experimental_protocol                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  notes
0   lobmann2012_01  Simvastatin   Glipizide          2              low          non_standard  Storage at 4 °C/0% RH (P2O5 desiccator). XRPD detected onset at day

## Vaihe 5 — Tallenna interim-CSV

In [12]:
df.to_csv(INTERIM_CSV_PATH, index=False)
print(f"[OK] Tallennettu: {INTERIM_CSV_PATH} ({len(df)} riviä)")
print("\nSEURAAVAT TOIMET:")
print("1. Tarkista yllä listatut needs_review / low-confidence -rivit käsin.")
print(f"2. Editoi tarvittaessa {RAW_JSON_PATH.name}, aja Vaihe 2-5 uudelleen.")
print("3. Kun OK, lisää lähde merge.py:n yhdistämislistaan ja päivitä master-CSV.")

[OK] Tallennettu: c:\Users\perhe\vscode\coamorphous\data\interim\lobmann_2012.csv (15 riviä)

SEURAAVAT TOIMET:
1. Tarkista yllä listatut needs_review / low-confidence -rivit käsin.
2. Editoi tarvittaessa lobmann_2012_raw.json, aja Vaihe 2-5 uudelleen.
3. Kun OK, lisää lähde merge.py:n yhdistämislistaan ja päivitä master-CSV.
